# Prediksi Permintaan Produk Retail — Pipeline Lengkap
**Kelompok:** Orang-orang Sukses | **Kelas:** DS-48-01 | **Telkom University 2026**

Pipeline ini mencakup:
1. Load & Inspeksi Dataset
2. Data Preprocessing
3. Exploratory Data Analysis (EDA)
4. Pembangunan & Evaluasi Model ML
5. Simpan Model Terbaik (untuk deployment API)


In [ ]:
# ============================================================
# 0. IMPORT LIBRARY
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)

print("Library berhasil diimpor.")


In [ ]:
# ============================================================
# 1. LOAD DATASET
# ============================================================
df = pd.read_csv("Warehouse_and_Retail_Sales.csv")

print(f"Dataset berhasil dimuat.")
print(f"Jumlah baris  : {df.shape[0]:,}")
print(f"Jumlah kolom  : {df.shape[1]}")
print(f"Kolom yang tersedia: {df.columns.tolist()}")

print("--- 5 Baris Pertama ---")
display(df.head())

print("--- Info Tipe Data ---")
df.info()

print("--- Statistik Deskriptif ---")
display(df.describe(include="all"))


In [ ]:
# ============================================================
# 2. DATA PREPROCESSING
# ============================================================

# 2.1 Missing values
missing = df.isnull().sum()
print("--- Missing Values per Kolom ---")
display(pd.DataFrame({"Jumlah": missing, "Persen (%)": (missing/len(df)*100).round(2)})[missing > 0])

if missing.sum() > 0:
    plt.figure(figsize=(10, 4))
    missing[missing > 0].plot(kind="bar", color="salmon")
    plt.title("Missing Values per Kolom")
    plt.tight_layout()
    plt.show()

# 2.2 Hapus duplikat
dup_count = df.duplicated().sum()
print(f"Jumlah baris duplikat: {dup_count}")
df = df.drop_duplicates()

# 2.3 Tangani missing values numerik (median) dan kategorik (modus)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ["float64", "int64"]:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

# 2.4 Filter nilai negatif pada target (kemungkinan retur)
target_col = "RETAIL SALES"
before = len(df)
df = df[df[target_col] >= 0]
print(f"Baris dengan RETAIL SALES negatif dihapus: {before - len(df)}")
print(f"Dataset setelah preprocessing: {len(df):,} baris")

print("--- Data Setelah Preprocessing ---")
display(df.head())


In [ ]:
# ============================================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

target_col = "RETAIL SALES"
cat_col = "ITEM TYPE"

# 3.1 Distribusi Target
print("--- 3.1 Distribusi RETAIL SALES ---")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[target_col].dropna(), bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Distribusi RETAIL SALES")
axes[1].boxplot(df[target_col].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor="steelblue", color="navy"))
axes[1].set_title("Boxplot RETAIL SALES")
plt.suptitle("Distribusi Target: RETAIL SALES", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(df[target_col].describe().round(2))

# 3.2 Penjualan per Kategori Produk
print("--- 3.2 Penjualan per Item Type ---")
cat_sales = df.groupby(cat_col)[target_col].sum().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cat_sales.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Total RETAIL SALES per Item Type")
axes[0].set_xlabel("Total Sales")
axes[1].pie(cat_sales[cat_sales > 0].values,
            labels=cat_sales[cat_sales > 0].index,
            autopct="%1.1f%%",
            colors=sns.color_palette("Set2", len(cat_sales)))
axes[1].set_title("Proporsi RETAIL SALES per Item Type")
plt.suptitle("Analisis Penjualan per Kategori", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 3.3 Tren Penjualan per Bulan
print("--- 3.3 Tren Penjualan per Bulan ---")
monthly = df.groupby("MONTH")[target_col].sum()
plt.figure(figsize=(12, 5))
plt.plot(monthly.index, monthly.values, marker="o", linewidth=2,
         color="steelblue", markerfacecolor="tomato", markersize=8)
plt.fill_between(monthly.index, monthly.values, alpha=0.15, color="steelblue")
plt.title("Tren Total RETAIL SALES per Bulan", fontsize=13, fontweight="bold")
plt.xlabel("Bulan")
plt.ylabel("Total RETAIL SALES")
plt.xticks(range(1, 13), ["Jan","Feb","Mar","Apr","Mei","Jun",
                           "Jul","Agu","Sep","Okt","Nov","Des"])
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# 3.4 Tren per Tahun
print("--- 3.4 Tren Penjualan per Tahun ---")
yearly = df.groupby("YEAR")[target_col].sum()
plt.figure(figsize=(10, 5))
yearly.plot(kind="bar", color="steelblue", edgecolor="white")
plt.title("Total RETAIL SALES per Tahun", fontsize=13, fontweight="bold")
plt.xlabel("Tahun")
plt.ylabel("Total RETAIL SALES")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 3.5 Korelasi fitur numerik
print("--- 3.5 Heatmap Korelasi ---")
num_df = df[["YEAR", "MONTH", "RETAIL SALES", "RETAIL TRANSFERS", "WAREHOUSE SALES"]]
plt.figure(figsize=(8, 6))
corr_matrix = num_df.corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Heatmap Korelasi Fitur Numerik", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 3.6 Deteksi Outlier
print("--- 3.6 Deteksi Outlier (IQR Method) ---")
Q1 = df[target_col].quantile(0.25)
Q3 = df[target_col].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df[target_col] < Q1 - 1.5*IQR) | (df[target_col] > Q3 + 1.5*IQR)]
print(f"Jumlah outlier: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")

In [ ]:
# ============================================================
# 4. PEMBANGUNAN MODEL MACHINE LEARNING
# ============================================================

# 4.1 Feature Engineering
print("--- 4.1 Persiapan Fitur ---")

df_model = df.copy()

# Encode ITEM TYPE
item_type_encoder = LabelEncoder()
df_model["ITEM TYPE"] = item_type_encoder.fit_transform(df_model["ITEM TYPE"].astype(str))
print(f"Item types: {list(item_type_encoder.classes_)}")

# Fitur yang digunakan
FEATURES = ["YEAR", "MONTH", "ITEM TYPE", "RETAIL TRANSFERS", "WAREHOUSE SALES"]
TARGET = "RETAIL SALES"

X = df_model[FEATURES]
y = df_model[TARGET]

print(f"Jumlah fitur: {X.shape[1]} | Jumlah data: {len(X):,}")

# 4.2 Split data (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training: {len(X_train):,} | Testing: {len(X_test):,}")

# 4.3 Scaling (untuk Linear Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# 4.4 Training & Evaluasi Semua Model
print("--- 4.4 Training Model ---")

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    if name == "Linear Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="r2")
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results.append({
        "Model": name, "MAE": round(mae, 4), "RMSE": round(rmse, 4),
        "R2": round(r2, 4), "CV R2 (mean)": round(cv_scores.mean(), 4),
        "_model": model, "_pred": y_pred
    })
    print(f"{name}: MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f} | CV R2={cv_scores.mean():.4f}")

results_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in results])
results_df = results_df.sort_values("RMSE").reset_index(drop=True)
print("--- Tabel Perbandingan Model ---")
display(results_df)

best_name = results_df.iloc[0]["Model"]
best_result = next(r for r in results if r["Model"] == best_name)
print(f"Model terbaik berdasarkan RMSE terendah: {best_name}")

In [ ]:
# 4.5 Visualisasi Perbandingan Metrik
print("--- 4.5 Visualisasi Perbandingan Model ---")

model_names = results_df["Model"].tolist()
colors = sns.color_palette("Set2", len(model_names))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, metric, label in zip(axes,
                              ["MAE", "RMSE", "R2"],
                              ["MAE (lebih kecil = lebih baik)",
                               "RMSE (lebih kecil = lebih baik)",
                               "R2 Score (lebih besar = lebih baik)"]):
    vals = results_df[metric].tolist()
    ax.barh(model_names, vals, color=colors)
    ax.set_title(label, fontweight="bold")
    for i, v in enumerate(vals):
        ax.text(v, i, f" {v:.3f}", va="center", fontsize=9)
plt.suptitle("Perbandingan Metrik Evaluasi Semua Model", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 4.6 Aktual vs Prediksi (Model Terbaik)
y_pred_best = best_result["_pred"]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].scatter(y_test, y_pred_best, alpha=0.4, color="steelblue", s=20)
min_v, max_v = min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())
axes[0].plot([min_v, max_v], [min_v, max_v], "r--", linewidth=2)
axes[0].set_title(f"Aktual vs Prediksi — {best_name}", fontweight="bold")
axes[0].set_xlabel("Aktual")
axes[0].set_ylabel("Prediksi")
residuals = y_test.values - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.4, color="tomato", s=20)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title(f"Residual Plot — {best_name}", fontweight="bold")
plt.tight_layout()
plt.show()

# 4.7 Feature Importance
if hasattr(best_result["_model"], "feature_importances_"):
    fi = pd.DataFrame({"Fitur": FEATURES, "Importance": best_result["_model"].feature_importances_})
    fi = fi.sort_values("Importance", ascending=False)
    plt.figure(figsize=(9, 5))
    sns.barplot(data=fi, x="Importance", y="Fitur", palette="viridis")
    plt.title(f"Feature Importance — {best_name}", fontweight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 5. SIMPAN MODEL TERBAIK (untuk Deployment API)
# ============================================================

Path("models").mkdir(exist_ok=True)

joblib.dump(best_result["_model"], "models/best_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(item_type_encoder, "models/item_type_encoder.pkl")

metadata = {
    "model_name": best_name,
    "target": TARGET,
    "features": FEATURES,
    "metrics": {
        "mae": float(results_df.iloc[0]["MAE"]),
        "rmse": float(results_df.iloc[0]["RMSE"]),
        "r2": float(results_df.iloc[0]["R2"]),
    },
    "item_types": [c for c in item_type_encoder.classes_ if c and c != "nan"],
}

with open("models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Model artifacts berhasil disimpan:")
print("  models/best_model.pkl")
print("  models/scaler.pkl")
print("  models/item_type_encoder.pkl")
print("  models/metadata.json")
print(f"Model terbaik: {best_name}")
print(f"  MAE  : {metadata['metrics']['mae']}")
print(f"  RMSE : {metadata['metrics']['rmse']}")
print(f"  R2   : {metadata['metrics']['r2']}")
print(f"Item types yang didukung: {metadata['item_types']}")
print("Model siap di-deploy ke FastAPI.")